# T06 · Kiểm tra và bàn giao dữ liệu P03

Notebook tái tạo ba bảng T05 từ kết quả T01, T02, T04 và BACI đã chuẩn bị. Các bước tiếp theo kiểm tra khóa ghép, độ phủ dữ liệu, số mẫu theo nhóm và tổng kim ngạch. Xem hướng dẫn chạy trong `docs/task56_update.md`.

Bản bàn giao gồm 398 mẫu khảo sát. Người thu được biểu diễn bằng mã để kiểm tra độ phủ và tỷ lệ đóng góp.


In [1]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "P03 Survey/scripts/t05_merge.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Mở notebook từ trong repo P03 đã giải nén.")
out = root / "P03 Survey/data_clean"
docs = root / "docs"
price_input = root / "P03 Survey/data_clean/survey_segment_All.csv"
hs_input_path = out.parent / "data_raw/hs/hs2022_final.csv"
baci_dir = root / "P03 Baci/data_clean"

subprocess.run(
    [sys.executable, str(root / "P03 Survey/scripts/t05_merge.py")],
    check=True,
)


## Kiểm tra khóa, giá trị và kết quả ghép

Các điều kiện bắt buộc gồm khóa duy nhất, 398 mã đầy đủ, mã HS hợp lệ, ngày năm hợp lệ và giá trị thương mại hữu hạn. Ô Task 4 trống và các cờ cần xác minh được ghi nhận để bàn giao.


In [2]:
def read(path):
    return pd.read_csv(path, dtype={"hs2": str, "record_id": str, "partner_code": str})

def require(condition, message):
    if not condition:
        raise ValueError(message)

def check_keys(frame, keys, name):
    require(frame[keys].notna().all().all(), f"{name}: thiếu khóa.")
    require(not frame.duplicated(keys).any(), f"{name}: trùng khóa.")

survey = read(out / "survey_final.csv")
base = read(out / "survey_clean.csv")
hs_input = read(hs_input_path)
segments = read(price_input)
survey_summary = read(out / "survey_hs_summary.csv")
baci_summary = read(out / "baci_hs_summary.csv")
products = read(baci_dir / "product_year.csv")
partners = read(baci_dir / "partner_year.csv")
trade = read(baci_dir / "trade_year.csv")

for name, frame in [("T01", base), ("T02", hs_input), ("T04", segments), ("T05", survey)]:
    check_keys(frame, ["record_id"], name)
require(len(base) == len(survey) == 398, "Số mẫu khảo sát khác 398.")
require(survey.record_id.equals(base.record_id), "Mất, thêm hoặc đổi thứ tự mã khảo sát.")
require(set(hs_input.record_id) == set(survey.record_id), "Bảng HS không phủ đúng mã khảo sát.")
require(set(segments.record_id) == set(survey.record_id), "T04 cần phủ đủ mã khảo sát.")
require(survey.category_clean.fillna("").str.strip().ne("").all(), "Có mẫu thiếu nhóm hàng.")
unpriced = survey.price_tier.fillna("").str.strip().eq("")
require(unpriced.sum() == 92, "Bản bàn giao dự kiến có 92 mẫu chưa xếp mức giá.")
require(survey.collector.str.fullmatch(r"C[0-9]{2}").fillna(False).all(),
        "Có mẫu thiếu mã người thu hoặc mã sai định dạng.")
require(pd.to_datetime(survey.survey_date, format="%Y-%m-%d", errors="coerce").notna().all(),
        "Có ngày khảo sát thiếu hoặc không hợp lệ.")
require(survey.loc[unpriced, "task4_note"].fillna("").str.strip().ne("").all(),
        "Có mẫu chưa xếp giá nhưng thiếu lý do T04.")
require(survey.origin_country.fillna("").str.strip().ne("").all(), "Có mẫu thiếu xuất xứ.")
prices = pd.to_numeric(survey.price_vnd, errors="coerce")
require(np.isfinite(prices).all() and prices.gt(0).all(), "Có giá khảo sát thiếu hoặc không hợp lệ.")

valid_hs = {f"{i:02}" for i in range(1, 98) if i != 77}
for name, frame in [("Khảo sát", survey), ("BACI sản phẩm", products), ("T05 BACI", baci_summary)]:
    invalid = frame.loc[~frame.hs2.isin(valid_hs)]
    require(invalid.empty, f"{name}: {len(invalid)} dòng thiếu hoặc sai HS2.")
require(set(survey.hs2).issubset(products.hs2), "Có Chapter khảo sát vắng trong BACI.")
summary_keys = ["hs2", "store", "origin_country", "category_clean", "price_tier"]
require(not survey_summary.duplicated(summary_keys).any(), "Tổng hợp khảo sát trùng nhóm.")
pd.testing.assert_series_equal(
    survey.groupby(summary_keys, dropna=False).size(),
    survey_summary.groupby(summary_keys, dropna=False).product_count.sum(),
    check_names=False,
)
require("trade_usd" not in survey, "Kim ngạch BACI bị đưa vào dòng mẫu khảo sát.")
keys = ["year", "hs2", "trade_flow"]
check_keys(products, keys, "BACI sản phẩm")
check_keys(baci_summary, keys, "T05 BACI")
check_keys(partners, ["year", "partner_code", "trade_flow"], "BACI đối tác")
check_keys(trade, ["year"], "BACI tổng theo năm")
require(partners.partner_code.str.fullmatch(r"[0-9]{1,3}").fillna(False).all(), "Sai mã đối tác.")
special_partner = partners.partner_code.eq("490") & partners.partner_iso3.eq("S19")
standard_iso = partners.partner_iso3.str.fullmatch(r"[A-Z]{3}").fillna(False)
require((standard_iso | special_partner).all(), "Sai hoặc thiếu mã ISO3/nhóm đối tác.")
print(f"Nhóm đối tác 490/S19 (Other Asia, nes): {special_partner.sum()} dòng.")
for name, frame in [("BACI sản phẩm", products), ("BACI đối tác", partners), ("T05 BACI", baci_summary)]:
    require(set(frame.year) == set(range(2005, 2025)), f"{name}: sai phạm vi năm.")
    require(set(frame.trade_flow) == {"Export", "Import"}, f"{name}: sai chiều thương mại.")
    values = pd.to_numeric(frame.trade_usd, errors="coerce")
    require(np.isfinite(values).all() and values.ge(0).all(), f"{name}: kim ngạch không hợp lệ.")

pd.testing.assert_frame_equal(
    products[baci_summary.columns].sort_values(keys).reset_index(drop=True),
    baci_summary.sort_values(keys).reset_index(drop=True),
    check_dtype=False,
    rtol=0,
    atol=0.01,
)
print(f"Khảo sát: {len(survey)} mã duy nhất, {survey.hs2.nunique()} chương HS, 0 mã HS thiếu.")
print(f"BACI: {len(products)} dòng sản phẩm, {len(partners)} dòng đối tác.")
print(f"Chưa xếp mức giá: {survey.price_tier.isna().sum()} mẫu.")


Nhóm đối tác 490/S19 (Other Asia, nes): 40 dòng.
Khảo sát: 398 mã duy nhất, 26 chương HS, 0 mã HS thiếu.
BACI: 3840 dòng sản phẩm, 6776 dòng đối tác.
Chưa xếp mức giá: 92 mẫu.


In [3]:
expected = trade[["year", "export_usd", "import_usd"]].melt(
    id_vars="year", var_name="trade_flow", value_name="trade_year_usd"
)
expected["trade_flow"] = expected.trade_flow.map({"export_usd": "Export", "import_usd": "Import"})
check_keys(expected, ["year", "trade_flow"], "trade_year")
require(set(expected.year) == set(range(2005, 2025)), "trade_year sai phạm vi năm.")
require(np.isfinite(expected.trade_year_usd).all() and expected.trade_year_usd.ge(0).all(),
        "trade_year có kim ngạch thiếu hoặc không hợp lệ.")
comparison = expected
for label, table in [("product", products), ("partner", partners), ("task5", baci_summary)]:
    col = f"{label}_usd"
    totals = table.groupby(["year", "trade_flow"], as_index=False).trade_usd.sum()
    comparison = comparison.merge(totals.rename(columns={"trade_usd": col}),
                                  on=["year", "trade_flow"], how="outer", validate="one_to_one")
    delta = comparison[col] - comparison.trade_year_usd
    require(delta.notna().all() and delta.abs().le(0.01).all(),
            f"{label}: lệch tổng kim ngạch quá 0,01 USD hoặc thiếu cặp năm × chiều.")
    print(f"{label}: khớp {len(delta)} cặp năm × chiều, lệch tối đa {delta.abs().max():,.2f} USD.")


product: khớp 40 cặp năm × chiều, lệch tối đa 0.00 USD.
partner: khớp 40 cặp năm × chiều, lệch tối đa 0.00 USD.
task5: khớp 40 cặp năm × chiều, lệch tối đa 0.00 USD.


In [4]:
records = [("490", "partner_iso3", "Mã nhóm BACI S19", "Giữ nhóm Other Asia, nes trong đối chiếu kim ngạch")]
for column in ["survey_time", "brand", "spec_text"]:
    for key in survey.loc[survey[column].isna(), "record_id"]:
        records.append((key, column, "Thiếu dữ liệu", "Giữ trống theo dữ liệu T01"))
for row in survey.loc[survey.verify_status.eq("Cần xác minh")].itertuples():
    records.append((row.record_id, row.verify_fields, "Cần xác minh", "Giữ cờ đối chiếu từ T01"))
for row in survey.loc[unpriced].itertuples():
    records.append((row.record_id, "price_tier", "Chưa xếp mức giá", row.task4_note))
collector_counts = survey.collector.value_counts().sort_index().rename("sample_count").to_frame()
collector_counts["share_pct"] = (collector_counts.sample_count / len(survey) * 100).round(2)
collector_counts["below_20_pct"] = collector_counts.sample_count * 5 < len(survey)
print(f"Đã xác định mã người thu cho {len(survey)} mẫu.")
print(collector_counts.to_string())
records.append(("", "hs2", "Chuẩn hóa mã", "Dùng mã dạng hai ký tự, giữ số 0 đầu"))
log = pd.DataFrame(records, columns=["record_id", "field", "issue", "action"])
t09_log = docs / "t09_cleaning_log_additions.csv"
if t09_log.is_file():
    additions = pd.read_csv(t09_log, dtype=str, keep_default_na=False)
    require(list(additions.columns) == list(log.columns), "Sai cấu trúc nhật ký T09.")
    log = pd.concat([log, additions], ignore_index=True)
log.to_csv(docs / "cleaning_log.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
counts = survey.store.value_counts().sort_values()
bars = axes[0].barh(counts.index, counts.values, color="#246a73")
axes[0].bar_label(bars, padding=4)
axes[0].set_xlim(0, counts.max() * 1.18)
axes[0].set_title("Số mẫu theo điểm khảo sát")
axes[0].set_xlabel("Số mẫu")
price_counts = [(~unpriced).sum(), unpriced.sum()]
bars = axes[1].bar(["Có mức giá", "Chưa xếp mức giá"], price_counts, color=["#246a73", "#cf8256"])
axes[1].bar_label(bars, padding=4)
axes[1].set_ylim(0, max(price_counts) * 1.18)
axes[1].set_title("Độ phủ phân mức giá")
axes[1].set_ylabel("Số mẫu")
fig.savefig(docs / "t06_quality.png", dpi=160, bbox_inches="tight")
plt.show()


Đã xác định mã người thu cho 398 mẫu.
           sample_count  share_pct  below_20_pct
collector                                       
C01                  83      20.85         False
C02                 105      26.38         False
C03                  70      17.59          True
C04                  70      17.59          True
C05                  70      17.59          True


## Bàn giao

Xem docs/data_cards.md để đọc nguồn, quy tắc và giới hạn. Kết quả T05 nằm trong P03 Survey/data_clean/. Log và biểu đồ T06 nằm trong docs/.
